In [1]:
import pandas as pd

df = pd.read_csv('/Users/jeremy/Documents/protests_chai/frames/truncated_samples.csv.gz')

df.columns

Index(['row_id', 'text-topic', 'text-topic-exp', 'text-entity-name',
       'text-entity-gender', 'text-entity-sentiment',
       'text-entity-sentiment-exp', 'text-generic-frame',
       'text-generic-frame-exp', 'text-issue-frame', 'text-issue-frame-exp',
       'uuid', 'title', 'img-generic-frame', 'img-frame-exp',
       'img-entity-name', 'img-entity-gender', 'img-entity-sentiment',
       'img-entity-sentiment-exp', 'image-url', 'gpt-topic', 'authors',
       'date_publish', 'description', 'language', 'maintext', 'source_domain',
       'url', 'political_leaning', 'month', 'keyword_matches',
       'keyword_match_count', 'word_count', 'label', 'protest_excerpt',
       'excerpt_length'],
      dtype='str')

In [ ]:
sample1 = df.sample(n=40, random_state=42)

remaining = df.drop(sample1.index)
sample2 = remaining.sample(n=40, random_state=45)
sample2 = sample2[['month', 'uuid', 'title', 'protest_excerpt', 'keyword_matches']]

sample2.to_csv('/Users/jeremy/Documents/protests_chai/frames/sample_1.csv', index=False)

#remaining = remaining.drop(sample2.index)
# for sample 3

In [33]:
sample2.head()

,month,uuid,title,protest_excerpt,keyword_matches
3711,2024-04-01_2024-04-30,817746e9-192f-4f41-81fc-6576fb8b1d46,Pro-Palestinian Student: 'Zionists Don't Deser...,The Israeli government and military should be ...,"['mob', 'protests', 'protests']"
5720,2023-10-01_2023-10-31,119d433c-3820-4d14-82b5-7af0b73d9d24,Australian police investigating pro-Palestinia...,Australian police announced Tuesday it has ope...,"['protest', 'protestors', 'Protest', 'protest'..."
514,2023-08-01_2023-08-31,659e1a63-db75-4fa8-9973-4c06f29c1148,"Hundreds rally in Niger, denouncing France, as...","NIAMEY, Niger (AP) — Hundreds of people rallie...","['demonstration', 'protesters', 'protest', 'pr..."
2587,2023-05-01_2023-05-31,8dbb8b86-43f4-4513-a9f5-1a75a5c173a7,Marine veteran in NYC subway chokehold death f...,"""Even if you're initially allowed to use force...","['PROTESTERS', 'demonstrations', 'protesters']"
1073,2023-08-01_2023-08-31,df34b27f-dd95-4afe-9806-1511dbccbc26,American Airlines Flight Attendants Say They W...,"“We are ready,” said Rosaland Groves, a 31-yea...","['demonstrations', 'picket']"


In [42]:
import re
import pandas as pd
import simpledorff


def parse_excel_column(text):
    """
    Paste a single Excel column (numbers or text labels).
    Works with line breaks or tabs copied directly from Excel.
    """
    return [
        x.strip()
        for x in text.strip().splitlines()
        if x.strip()
    ]

def calculate_agreement(coder_a_text, coder_b_text, label_name="label"):
    coder_a = parse_excel_column(coder_a_text)
    coder_b = parse_excel_column(coder_b_text)

    if len(coder_a) != len(coder_b):
        raise ValueError(
            f"{label_name}: unequal lengths "
            f"({len(coder_a)} vs. {len(coder_b)})"
        )

    if len(coder_a) == 0:
        raise ValueError(f"{label_name}: no values were found.")

    wide = pd.DataFrame({
        "unit": range(len(coder_a)),
        "coder_a": coder_a,
        "coder_b": coder_b
    })

    raw_agreement = (wide["coder_a"] == wide["coder_b"]).mean()

    long = wide.melt(
        id_vars="unit",
        value_vars=["coder_a", "coder_b"],
        var_name="annotator",
        value_name="label"
    )

    alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
        long,
        experiment_col="unit",
        annotator_col="annotator",
        class_col="label"
    )

    return {
        "field": label_name,
        "n": len(wide),
        "raw_agreement": raw_agreement,
        "krippendorff_alpha": float(alpha)
    }

In [44]:
jeremy_title_frame = """
violence
violence
widespread_support
grievance
widespread_support
violence
disruption
confrontation
grievance
violence
grievance
disruption
disruption
violence
disruption
widespread_support
grievance
confrontation
confrontation
grievance
confrontation
violence
violence
widespread_support
residual
impactful
disruption
grievance
disruption
property_damage
residual
violence
residual
grievance
confrontation
grievance
confrontation
violence
disruption
peaceful
"""

julia_title_frame = """
violence
confrontation
confrontation
confrontation
widespread_support
peaceful
widespread_support
confrontation
confrontation
confrontation
peaceful
peaceful
disruption
violence
impactful
widespread_support
grievance
residual
disruption
widespread_support
disruption
disruption
violence
grievance
residual
impactful
disruption
disruption
disruption
property_damage
property_damage
violence
grievance
residual
confrontation
grievance
residual
violence
disruption
widespread_support
"""

result = calculate_agreement(
    jeremy_title_frame,
    julia_title_frame,
    label_name="body_frame"
)

print(result)

with open("agreement_results.txt", "a") as f:
    f.write(
        f"{result['field']}: "
        f"n={result['n']}, "
        f"Raw Agreement={result['raw_agreement']:.3f}, "
        f"Krippendorff Alpha={result['krippendorff_alpha']:.3f}\n"
    )

{'field': 'body_frame', 'n': 40, 'raw_agreement': np.float64(0.45), 'krippendorff_alpha': 0.3682297346419484}
